# Anomaly detection: inspect two scores by hand

Companion to HTB Academy Module 290, Section 12. These are invented *arithmetic inputs*, not fitted models or security verdicts. Read $E(h(x))$ as **average path length for x**, $c(n)$ as **c of n**, $\operatorname{lrd}(p)$ as **local reachability density of p**, and $\operatorname{LOF}(p)$ as **L-O-F of p**.

A point anomaly differs alone; a contextual anomaly depends on conditions such as time; a collective anomaly appears only when events are considered together. A numerical score still needs context and a validated alert threshold.

## Isolation Forest's path-length score

Read $s(x)=2^{-E(h(x))/c(n)}$ as **score of x equals two raised to the negative average path length divided by the normalization factor**. Here `10` is chosen only to make arithmetic simple: it is **not** a derived value of $c(n)$ for a real dataset. Shorter paths yield higher scores. This code does not fit or simulate an Isolation Forest.

In [ ]:
# The two invented observations differ only in average isolation depth.
normalizer = 10.0  # Illustrative c(n), not calculated from a dataset size.
short_path = 2.0  # E(h(x)) averaged over imagined trees.
long_path = 10.0

def isolation_score(average_path, c_n):
    """Implement HTB's s(x) = 2^(-E(h(x))/c(n)) formula."""
    if c_n <= 0 or average_path < 0:
        raise ValueError('Need a positive normalizer and nonnegative path length')
    return 2 ** (-average_path / c_n)

short_score = isolation_score(short_path, normalizer)
long_score = isolation_score(long_path, normalizer)
print(f'short path: {short_score:.3f}; long path: {long_score:.3f}')
assert abs(short_score - 0.8705505633) < 1e-9
assert long_score == 0.5
assert short_score > long_score  # Easier isolation -> stronger anomaly signal.

## Reachability distance, local density, then LOF

Read $\operatorname{reach\_dist}(p,o)=\max(d(p,o),\operatorname{kdist}(o))$ as **the larger of p's actual distance to neighbor o and o's k-distance**. Read $\operatorname{lrd}(p)=1/(\sum \operatorname{reach\_dist}/k)$ as **one divided by mean reachability distance**. Finally, read $\operatorname{LOF}(p)=(\sum\operatorname{lrd}(o)/k)/\operatorname{lrd}(p)$ as **neighbor mean density divided by p's density**. The example supplies neighbor distances and densities directly; a complete LOF implementation would have to find neighbors and compute each one's density from data.

In [ ]:
# Two supplied neighbors (k=2). Distances use the same illustrative units.
actual_distances = [1.0, 5.0]  # d(p,o) for each neighbor o.
neighbor_k_distances = [3.0, 3.0]  # k-distance of each neighbor o.
neighbor_lrd = [0.5, 0.5]  # Their precomputed local densities.
k = len(actual_distances)
assert len(neighbor_k_distances) == len(neighbor_lrd) == k

# Formula 1: reach_dist(p,o) = max(actual distance, neighbor's k-distance).
reach_distances = [max(actual, k_distance) for actual, k_distance in zip(actual_distances, neighbor_k_distances)]
# Formula 2: lrd(p) = 1 / ((sum of reachability distances) / k).
average_reach_distance = sum(reach_distances) / k
if average_reach_distance <= 0:
    raise ValueError('Local reachability density needs positive mean distance')
point_lrd = 1 / average_reach_distance
# Formula 3: LOF(p) = (mean lrd of the neighbors) / lrd(p).
average_neighbor_lrd = sum(neighbor_lrd) / k
lof = average_neighbor_lrd / point_lrd
print('reachability distances:', reach_distances)
print('point density:', point_lrd, 'neighbor mean density:', average_neighbor_lrd, 'LOF:', lof)
assert reach_distances == [3.0, 5.0]
assert point_lrd == 0.25
assert lof == 2.0

## Interpret carefully

The short-path observation scores higher under the Isolation Forest formula. The point with LOF 2 has half its neighbors' average local density. Neither result says **80% likely malicious** or **twice as likely to be an attack**. Real detection requires feature design, representative baseline data, a threshold selected against labeled reviews, and monitoring for drift and false alerts.